#### Loading the Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

#### Loading the Feature-Engineered Dataset

In [2]:
df = pd.read_csv(
    '../data/processed/feature_engineered_data.csv'
)

df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Region,TrafficType,VisitorType,Weekend,Revenue,Total_Pages_Viewed,Total_Browsing_Duration,Product_Page_Share,Avg_Duration_Per_Page,Product_Duration_Share
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,1,1,Returning_Visitor,False,False,1,0.000000,1.0,0.000000,0.0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,...,1,2,Returning_Visitor,False,False,2,64.000000,1.0,32.000000,1.0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,9,3,Returning_Visitor,False,False,1,0.000000,1.0,0.000000,0.0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,...,2,4,Returning_Visitor,False,False,2,2.666667,1.0,1.333333,1.0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,...,1,4,Returning_Visitor,True,False,10,627.500000,1.0,62.750000,1.0


#### Creating the Modeling Dataset

In [3]:
df_model = df.copy()

#### Separating Features and Target

In [4]:
X = df_model.drop(
    'Revenue',
    axis=1
)

y = df_model['Revenue']

#### Checking the Feature and Target Shapes

In [5]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (12205, 22)
Target shape: (12205,)


#### Creating the Train-Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

#### Checking the Split Shapes

In [7]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (9764, 22)
X_test shape: (2441, 22)
y_train shape: (9764,)
y_test shape: (2441,)


#### Checking Target Distribution

In [8]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training target distribution:
Revenue
False    0.843712
True     0.156288
Name: proportion, dtype: float64

Testing target distribution:
Revenue
False    0.843507
True     0.156493
Name: proportion, dtype: float64


#### Creating the Training Dataset

In [9]:
train_data = X_train.copy()
train_data['Revenue'] = y_train.values

#### Creating the Testing Dataset

In [10]:
test_data = X_test.copy()
test_data['Revenue'] = y_test.values

#### Saving the Training Dataset

In [11]:
train_data.to_csv(
    '../data/processed/train_data.csv',
    index=False
)

#### Saving the Testing Dataset

In [12]:
test_data.to_csv(
    '../data/processed/test_data.csv',
    index=False
)

#### Checking the Saved Dataset Shapes

In [13]:
print("Training dataset shape:", train_data.shape)
print("Testing dataset shape:", test_data.shape)

Training dataset shape: (9764, 23)
Testing dataset shape: (2441, 23)


#### Identifying Numerical and Categorical Features

In [14]:
numerical_features = X_train.select_dtypes(
    include=['int64', 'float64', 'bool']
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=['object', 'category', 'str']
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend', 'Total_Pages_Viewed', 'Total_Browsing_Duration', 'Product_Page_Share', 'Avg_Duration_Per_Page', 'Product_Duration_Share']

Categorical features:
['Month', 'VisitorType']


OperatingSystems, Browser, Region, and TrafficType should be treated as categorical, not numerical. They are encoded category IDs, not quantities.

Also, Weekend is a binary feature, so we'll handle it separately as numeric/binary.

#### Defining Numerical and Categorical Features

In [15]:
numerical_features = [
    'Administrative',
    'Administrative_Duration',
    'Informational',
    'Informational_Duration',
    'ProductRelated',
    'ProductRelated_Duration',
    'BounceRates',
    'ExitRates',
    'PageValues',
    'SpecialDay',
    'Total_Pages_Viewed',
    'Total_Browsing_Duration',
    'Product_Page_Share',
    'Avg_Duration_Per_Page',
    'Product_Duration_Share'
]

categorical_features = [
    'Month',
    'OperatingSystems',
    'Browser',
    'Region',
    'TrafficType',
    'VisitorType'
]

binary_features = [
    'Weekend'
]

#### Checking the Feature Groups

In [16]:
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Binary features:", len(binary_features))

print(
    "\nTotal features:",
    len(numerical_features)
    + len(categorical_features)
    + len(binary_features)
)

Numerical features: 15
Categorical features: 6
Binary features: 1

Total features: 22


#### Importing Preprocessing Libraries

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#### Creating the Preprocessing Pipeline

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'numerical',
            StandardScaler(),
            numerical_features
        ),
        (
            'categorical',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_features
        ),
        (
            'binary',
            'passthrough',
            binary_features
        )
    ]
)

#### Importing SMOTE and Pipeline

In [19]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

#### Creating the SMOTE Configuration

In [20]:
smote = SMOTE(
    random_state=42
)

#### Baseline Model

In [21]:
from sklearn.linear_model import LogisticRegression

Creating the Logistic Regression Pipeline

In [22]:
logistic_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('smote', smote),
        (
            'model',
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

Training the Logistic Regression Model

In [23]:
logistic_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[bool](2,)","[False, True]"
feature_names_in_,"ndarray[object](22,)","['Administrative','Administrative_Duration','Informational',..., 'Product_Page_Share','Avg_Duration_Per_Page','Product_Duration_Share']"
n_features_in_,int,22
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3


Creating the Baseline Prediction

In [24]:
baseline_prediction = y_train.mode()[0]

y_test_baseline = np.full(
    len(y_test),
    baseline_prediction
)

Checking the Baseline Prediction

In [25]:
print("Baseline prediction:", baseline_prediction)
print("Baseline predictions:", len(y_test_baseline))

Baseline prediction: False
Baseline predictions: 2441


Baseline is established: it predicts False for every session.

Now we move to the first real model and, importantly, start our MLflow experiment tracking.

#### Setting the MLflow Experiment

In [26]:
import mlflow
import mlflow.sklearn

Setting the MLflow Experiment

In [27]:
mlflow.set_experiment(
    "E-commerce Conversion Dynamics"
)

<Experiment: artifact_location=('file:///d:/Projects/ML and Data Science/E-Commerce Conversion '
 'Dynamics/notebooks/mlruns/1'), creation_time=1789318046291, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789318046291, lifecycle_stage='active', name='E-commerce Conversion Dynamics', tags={}, trace_location=None, workspace='default'>

Configuring MLflow Tracking

In [28]:
mlflow.set_tracking_uri(
    "sqlite:///../mlflow.db"
)

mlflow.set_experiment(
    "E-commerce Conversion Dynamics"
)

<Experiment: artifact_location=('file:///d:/Projects/ML and Data Science/E-Commerce Conversion '
 'Dynamics/notebooks/mlruns/1'), creation_time=1789318954296, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789318954296, lifecycle_stage='active', name='E-commerce Conversion Dynamics', tags={}, trace_location=None, workspace='default'>

Checking the MLflow Tracking Location

In [29]:
print(mlflow.get_tracking_uri())

sqlite:///../mlflow.db


Checking the MLflow Experiment

In [30]:
experiment = mlflow.get_experiment_by_name(
    "E-commerce Conversion Dynamics"
)

print("Experiment ID:", experiment.experiment_id)
print("Experiment Name:", experiment.name)

Experiment ID: 1
Experiment Name: E-commerce Conversion Dynamics


Starting the Logistic Regression MLflow Run

In [31]:
with mlflow.start_run(run_name="Logistic Regression"):

    logistic_pipeline.fit(
        X_train,
        y_train
    )

    mlflow.log_param(
        "model",
        "Logistic Regression"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

Saving the Logistic Regression Pipeline

In [32]:
from joblib import dump

dump(
    logistic_pipeline,
    '../models/logistic_regression_pipeline.joblib'
)

['../models/logistic_regression_pipeline.joblib']

#### Ridge Classifier

In [33]:
from sklearn.linear_model import RidgeClassifier

In [34]:
## Creating the Ridge Classifier Pipeline

ridge_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('smote', smote),
        (
            'model',
            RidgeClassifier(
                random_state=42
            )
        )
    ]
)

In [35]:
## Starting the Ridge Classifier MLflow Run

with mlflow.start_run(run_name="Ridge Classifier"):

    ridge_pipeline.fit(
        X_train,
        y_train
    )

    mlflow.log_param(
        "model",
        "Ridge Classifier"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

In [36]:
## Saving the Ridge Classifier Pipeline

dump(
    ridge_pipeline,
    '../models/ridge_classifier_pipeline.joblib'
)

['../models/ridge_classifier_pipeline.joblib']

Decision Tree

In [38]:
## Loading the Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier


In [39]:
## Creating the Decision Tree Pipeline

decision_tree_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('smote', smote),
        (
            'model',
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

In [41]:
## Starting the Decision Tree MLflow Run

with mlflow.start_run(run_name="Decision Tree Classifier"):

    decision_tree_pipeline.fit(
        X_train,
        y_train
    )

    mlflow.log_param(
        "model",
        "Decision Tree Classifier"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

In [40]:
## Saving the Decision Tree Pipeline

dump(
    decision_tree_pipeline,
    '../models/decision_tree_classifier_pipeline.joblib'
)

['../models/decision_tree_classifier_pipeline.joblib']

Random Forest

In [42]:
from sklearn.ensemble import RandomForestClassifier

In [43]:
random_forest_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('smote', smote),
        (
            'model',
            RandomForestClassifier(
                random_state=42
            )
        )
    ]
)

In [44]:
with mlflow.start_run(run_name="Random Forest Classifier"):

    random_forest_pipeline.fit(
        X_train,
        y_train
    )

    mlflow.log_param(
        "model",
        "Random Forest Classifier"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

In [45]:
dump(
    random_forest_pipeline,
    '../models/random_forest_classifier_pipeline.joblib'
)

['../models/random_forest_classifier_pipeline.joblib']

Loading RandomizedSearchCV (for hyperparameter tunning)

In [46]:
## Loading the Hyperparameter Search

from sklearn.model_selection import RandomizedSearchCV

In [47]:
## Defining the Random Forest Parameter Grid

random_forest_param_grid = {
    'model__n_estimators': [200, 300, 500],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}

In [48]:
## Creating the Random Forest Hyperparameter Search

random_forest_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=random_forest_param_grid,
    n_iter=15,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [49]:
## Starting the Tuned Random Forest MLflow Run

with mlflow.start_run(run_name="Tuned Random Forest Classifier"):

    random_forest_search.fit(
        X_train,
        y_train
    )

    mlflow.log_param(
        "model",
        "Random Forest Classifier"
    )

    mlflow.log_param(
        "tuning",
        "RandomizedSearchCV"
    )

    mlflow.log_param(
        "n_iter",
        15
    )

    mlflow.log_param(
        "cv",
        5
    )

    mlflow.log_param(
        "scoring",
        "f1"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

Fitting 5 folds for each of 15 candidates, totalling 75 fits


In [50]:
## Checking the Best Random Forest Parameters

print(
    "Best Parameters:",
    random_forest_search.best_params_
)

Best Parameters: {'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 30}


In [51]:
## Checking the Best Cross-Validation Score

print(
    "Best Cross-Validation F1 Score:",
    random_forest_search.best_score_
)

Best Cross-Validation F1 Score: 0.6896357351152714


In [53]:
with mlflow.start_run(
    run_name="Tuned Random Forest Classifier"
):

    mlflow.log_param(
        "model",
        "Random Forest Classifier"
    )

    mlflow.log_param(
        "tuning",
        "RandomizedSearchCV"
    )

    mlflow.log_param(
        "n_iter",
        15
    )

    mlflow.log_param(
        "cv",
        5
    )

    mlflow.log_param(
        "scoring",
        "f1"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

    mlflow.log_params(
        random_forest_search.best_params_
    )

    mlflow.log_metric(
        "best_cv_f1",
        random_forest_search.best_score_
    )

    mlflow.sklearn.log_model(
        random_forest_search.best_estimator_,
        "model",
        skops_trusted_types=[
            "imblearn.over_sampling._smote.base.SMOTE",
            "imblearn.pipeline.Pipeline"
        ]
    )

2026/09/14 22:52:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [54]:
## Saving the Tuned Random Forest Pipeline

dump(
    random_forest_search.best_estimator_,
    '../models/tuned_random_forest_classifier_pipeline.joblib'
)

['../models/tuned_random_forest_classifier_pipeline.joblib']

Extra Trees

In [55]:
from sklearn.ensemble import ExtraTreesClassifier

In [56]:
extra_trees_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('smote', smote),
        (
            'model',
            ExtraTreesClassifier(
                random_state=42
            )
        )
    ]
)

In [57]:
with mlflow.start_run(
    run_name="Extra Trees Classifier"
):

    extra_trees_pipeline.fit(
        X_train,
        y_train
    )

    mlflow.log_param(
        "model",
        "Extra Trees Classifier"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

In [58]:
dump(
    extra_trees_pipeline,
    '../models/extra_trees_classifier_pipeline.joblib'
)

['../models/extra_trees_classifier_pipeline.joblib']

Tuning the Extra Trees Classifier

In [59]:
extra_trees_param_grid = {
    'model__n_estimators': [200, 300, 500],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}

In [60]:
extra_trees_search = RandomizedSearchCV(
    estimator=extra_trees_pipeline,
    param_distributions=extra_trees_param_grid,
    n_iter=15,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-2,
    verbose=1
)

In [61]:
extra_trees_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-2
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can b

In [62]:
print(
    "Best Parameters:"
)

print(
    extra_trees_search.best_params_
)

print(
    "\nBest CV F1 Score:",
    extra_trees_search.best_score_
)

Best Parameters:
{'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': None}

Best CV F1 Score: 0.579916672807616


In [63]:
## Saving the Tuned Extra Trees Pipeline

dump(
    extra_trees_search.best_estimator_,
    '../models/tuned_extra_trees_classifier_pipeline.joblib'
)

['../models/tuned_extra_trees_classifier_pipeline.joblib']

In [64]:
## Logging the Tuned Extra Trees to MLflow

with mlflow.start_run(
    run_name="Tuned Extra Trees Classifier"
):

    mlflow.log_param(
        "model",
        "Extra Trees Classifier"
    )

    mlflow.log_param(
        "tuning",
        "RandomizedSearchCV"
    )

    mlflow.log_param(
        "n_iter",
        15
    )

    mlflow.log_param(
        "cv",
        5
    )

    mlflow.log_param(
        "scoring",
        "f1"
    )

    mlflow.log_param(
        "smote",
        "SMOTE"
    )

    mlflow.log_param(
        "random_state",
        42
    )

    mlflow.log_params(
        extra_trees_search.best_params_
    )

    mlflow.log_metric(
        "best_cv_f1",
        extra_trees_search.best_score_
    )

    mlflow.sklearn.log_model(
        extra_trees_search.best_estimator_,
        "model",
        skops_trusted_types=[
            "imblearn.over_sampling._smote.base.SMOTE",
            "imblearn.pipeline.Pipeline"
        ]
    )

2026/09/14 23:05:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
